# 04 Second pass: a seeded 20 percent sample coded again, and agreement before any disagreement is resolved

Design log.
- Plan v2 A6: a random 20 percent of reports, drawn by a seeded script and listed in `reports/second_pass_ids.txt`, is coded again from the PDF without sight of the first coding. Agreement is reported per field type before disagreements are resolved. Fields below 0.8 exact agreement, or kappa below 0.6, are re-coded in full. With one coder this is a consistency check and is named as such.
- This notebook runs before `04_full_extraction`, so the agreement figures are in the repository before any record is changed.
- The sample: every included study in `reports/defences.csv` (124) is ranked by the SHA-256 of "2026|slug" and the first ceil(0.2 × N) = 25 are kept. A study that later leaves the frame does not reorder the others, so a full-text exclusion does not redraw the sample. All 25 sampled studies already have records.
- What was coded again, by the field types of plan A6:
  - categorical: the report's role; coverage of R1, R2clean, R2attack, R3ns, R3cp and R3safe; per defence: origin, isolation, signal dependence, trains_instruction_pair and trains_carry_through (exact agreement and Cohen's kappa, missing values counted as a category);
  - multi-label: axis 1 and axis 2 per defence, and the set of defences coded per report at the method level (Jaccard overlap);
  - numeric: up to six printed values per report, chosen by hash rank from the first coding and given to the second coder as an address card (page, table, arm, benchmark, metric and unit) without the value; 146 values in all (AdverShield-LLM has two printed values); exact match within rounding;
  - not scored: free text (names, anchors, notes). Comparisons and claims are not coded again: their conditions rest on evaluation fields, and every decisive value is checked against its page in `04_full_extraction`.
- Matching: a second-pass defence entry names the method and lists its base models. `reports/second_pass_matches.yaml` links each entry to the first-pass defence records of the same method; it was written after the second coding was frozen, by method name and base model only, and changes no coded value. Each matched first-pass record is compared field by field with its entry.
- Disclosure: the second coding was done by the same coder. Before it, one-line batch notes had been seen for five of the 25 reports (explosive_arxiv2026, advershield_jtsi2026, municipal_sauc2026, codt_arxiv2025, securecai_arxiv2026). The notes name the tables and arms coded and, for AdverShield-LLM, that its attack success was coded as a proxy measure. No first-pass field value was read before the second coding was frozen.
- What the dry run showed (no model calls; the numbers are recomputed below from the committed files):
  - all 146 printed values re-read from their addresses agree with the first coding;
  - role, coverage of R2clean and R2attack, defence origin, both training-target fields, axis 2 and the defence set meet the threshold;
  - seven fields fall below it: coverage of R1, R3ns, R3cp and R3safe; defence isolation; signal dependence; axis 1. For R3ns and R3safe, 24 of 25 reports agree and kappa is still near zero, because almost every report carries the same code and chance agreement is near 1. The rule is applied as written.
- Next: amend the coding rules where the disagreements show a gap, re-code the seven fields in every record (notebook 04r), code the last two studies (batch 04l), then `04_full_extraction`.
- CPU only; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
try:
    importlib.import_module("yaml")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 1. The extraction module and its tests (plan v2 B4: `src/extract.py`)

In [ ]:
%%writefile src/extract.py
"""Record files to CSV tables, the second-pass sample, value cards and agreement statistics (plan v2, B4).

Notebook 04 writes this module. Nothing here calls a model or the network."""
from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter, defaultdict

SEED = 2026
SHARE = 0.2
CARDS_PER_REPORT = 6
QUANTITIES = ["R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe"]
DEF_CATEGORICAL = ["origin", "isolation", "signal_dependence", "trains_instruction_pair", "trains_carry_through"]
DEF_MULTI = ["axis1", "axis2"]
EXACT_MIN, KAPPA_MIN, JACCARD_MIN = 0.8, 0.6, 0.8
MISSING = "(missing)"


def rank_key(seed, *parts) -> str:
    return hashlib.sha256("|".join([str(seed), *map(str, parts)]).encode()).hexdigest()


def second_pass_sample(included_slugs, seed: int = SEED, share: float = SHARE) -> list[str]:
    """The seeded sample: every included study ranked by a hash of the seed and its slug, the first
    ceil(share * N) kept. Removing a study from the frame does not reorder the others."""
    slugs = sorted(set(included_slugs))
    n = math.ceil(share * len(slugs))
    return sorted(slugs, key=lambda s: rank_key(seed, s))[:n]


def value_cards(rec: dict, k: int = CARDS_PER_REPORT, seed: int = SEED) -> list[dict]:
    """Up to k printed results per report, chosen by hash rank, given by their address only (never the value)."""
    rid = rec["report"]["id"]
    evs = {e["id"]: e for e in rec.get("evaluations") or []}
    dfs = {d["id"]: d for d in rec.get("defences") or []}
    printed = [r for r in rec.get("results") or [] if r.get("value_source") != "figure_only" and r.get("value") not in (None, "")]
    printed.sort(key=lambda r: rank_key(seed, rid, r["id"]))
    cards = []
    for r in printed[:k]:
        e = evs[r["evaluation_id"]]
        d = dfs.get(e["defence_id"])
        arm = f"{d['name']} on {d['base_checkpoint']}" if d else f"no defence ({str(e['defence_id'])[5:]}, {e.get('baseline_kind', '')})"
        extra = "; ".join(f"{f}: {e[f]}" for f in ("split", "run", "injection_position", "attacker_budget", "subset_of") if e.get(f))
        cards.append({"report": rid, "result": r["id"], "page": r.get("page"), "location": r.get("location", ""),
                      "arm": arm, "benchmark": e["benchmark"], "attacker": e["attacker_type"], "judge": e["judge"],
                      "metric": r["metric_as_printed"], "quantity": r["quantity"], "unit": r["unit"], "detail": extra})
    return cards


def _cell(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return ";".join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False)
    return v


def build_tables(records: dict, schema: dict, grade) -> dict:
    """The five extraction tables (defences, evaluations, results, comparisons, claims) and a report index, with
    ids made global as <report_id>:<local_id>. Derived columns: the defence each result belongs to, whether an
    evaluation is the original or a later measurement, and the grade of each comparison."""
    k = schema["kinds"]
    out = {name: [] for name in ("reports", "defences", "evaluations", "results", "comparisons", "claims")}
    for rid in sorted(records):
        rec = records[rid]
        rep, cov = rec["report"], rec.get("coverage") or {}
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        out["reports"].append({**{f: _cell(rep.get(f)) for f in k["report"]["fields"]},
                               **{f"cov_{q}": cov.get(q, "") for q in QUANTITIES},
                               "cov_where": _cell(cov.get("where")),
                               **{f"n_{x}": len(rec.get(x) or []) for x in ("defences", "evaluations", "results", "comparisons", "claims")}})
        for d in rec.get("defences") or []:
            out["defences"].append({"report_id": rid, "defence_uid": f"{rid}:{d['id']}", **{f: _cell(d.get(f)) for f in k["defences"]["fields"]}})

        def duid(did):
            did = str(did)
            return did if did.startswith("none:") or ":" in did else f"{rid}:{did}"

        def measurement(e):
            d = dfs.get(e["defence_id"])
            if d is None:
                return "undefended" if str(e["defence_id"]).startswith("none:") else "other_record"
            if d.get("origin") == "this_report" and e.get("measured_by") == rid:
                return "original"
            if d.get("origin") == "this_report":
                return "original_reprinted"
            return "later"

        for e in rec.get("evaluations") or []:
            out["evaluations"].append({"report_id": rid, "evaluation_uid": f"{rid}:{e['id']}", "defence_uid": duid(e["defence_id"]),
                                       "measurement": measurement(e), **{f: _cell(e.get(f)) for f in k["evaluations"]["fields"]}})
        for r in rec.get("results") or []:
            e = evs.get(r["evaluation_id"], {})
            out["results"].append({"report_id": rid, "result_uid": f"{rid}:{r['id']}", "evaluation_uid": f"{rid}:{r['evaluation_id']}",
                                   "defence_uid": duid(e.get("defence_id", "")), "measurement": measurement(e) if e else "",
                                   **{f: _cell(r.get(f)) for f in k["results"]["fields"]}})
        for c in rec.get("comparisons") or []:
            out["comparisons"].append({"report_id": rid, "comparison_uid": f"{rid}:{c['id']}", "left_uid": f"{rid}:{c['left']}",
                                       "right_uid": f"{rid}:{c['right']}", "grade": grade(c),
                                       **{f: _cell(c.get(f)) for f in k["comparisons"]["fields"]}})
        for cl in rec.get("claims") or []:
            row = {"report_id": rid, "claim_uid": f"{rid}:{cl['id']}", **{f: _cell(cl.get(f)) for f in k["claims"]["fields"]}}
            row["supported_by"] = ";".join(f"{rid}:{s}" for s in cl.get("supported_by") or [])
            row["contradicted_by"] = ";".join(f"{rid}:{s}" for s in cl.get("contradicted_by") or [])
            out["claims"].append(row)
    fields = {name: (list(rows[0].keys()) if rows else []) for name, rows in out.items()}
    return {name: (rows, fields[name]) for name, rows in out.items()}


def cross_record_problems(records: dict, included_slugs, page_counts: dict | None = None) -> list[str]:
    """Checks that need the whole corpus: one record per included study, references to other reports resolve,
    every record read in full, and (when page counts are given) every page within its PDF."""
    probs = []
    by_slug = defaultdict(list)
    for rid, rec in records.items():
        by_slug[rec["report"]["slug"]].append(rid)
    inc = set(included_slugs)
    for s in sorted(inc - set(by_slug)):
        probs.append(f"included study {s} has no record")
    for s in sorted(set(by_slug) - inc):
        probs.append(f"record(s) {by_slug[s]} belong to {s}, which is not an included study")
    for s, rids in by_slug.items():
        if len(rids) > 1:
            probs.append(f"included study {s} has {len(rids)} records: {rids}")
    defences = {f"{rid}:{d['id']}" for rid, rec in records.items() for d in rec.get("defences") or []}
    for rid, rec in records.items():
        if rec["report"].get("read") != "F":
            probs.append(f"{rid}: read is {rec['report'].get('read')!r}, not F")
        for e in rec.get("evaluations") or []:
            mb = e.get("measured_by")
            if mb != rid and mb not in records:
                probs.append(f"{rid}:{e['id']}: measured_by {mb!r} is neither this report nor another record")
            sc = e.get("same_checkpoint_as")
            if sc and sc not in defences:
                probs.append(f"{rid}:{e['id']}: same_checkpoint_as {sc!r} is not a defence in the corpus")
            did = str(e.get("defence_id", ""))
            if ":" in did and not did.startswith("none:") and did not in defences:
                probs.append(f"{rid}:{e['id']}: defence_id {did!r} is not a defence in the corpus")
        if page_counts and rid in page_counts:
            n = page_counts[rid]
            for kind in ("defences", "evaluations", "results", "comparisons", "claims"):
                for row in rec.get(kind) or []:
                    pages = [row.get("page")] + list((row.get("pages") or {}).values())
                    for p in pages:
                        if isinstance(p, int) and p > n:
                            probs.append(f"{rid}:{kind}:{row.get('id')}: page {p} beyond the PDF's {n} pages")
    return probs


def validate_second_pass(doc: dict, schema: dict, cards: list[dict], first: dict, match_line: list) -> list[str]:
    """Allowed values in a second-pass document, value answers for exactly the drawn cards, and match lines that
    point at first-pass defences of the same report."""
    k, rid, out = schema["kinds"], doc.get("report"), []
    if doc.get("role") not in k["report"]["fields"]["role"]["values"]:
        out.append(f"{rid}: role {doc.get('role')!r} not allowed")
    for q in QUANTITIES:
        if (doc.get("coverage") or {}).get(q) not in k["coverage"]["fields"][q]["values"]:
            out.append(f"{rid}: coverage {q} {(doc.get('coverage') or {}).get(q)!r} not allowed")
    for e in doc.get("defences") or []:
        for f in DEF_CATEGORICAL:
            if e.get(f) not in k["defences"]["fields"][f]["values"]:
                out.append(f"{rid}: {e.get('name')}: {f} {e.get(f)!r} not allowed")
        for f in DEF_MULTI:
            vals = e.get(f) or []
            if not vals or any(v not in k["defences"]["fields"][f]["values"] for v in vals):
                out.append(f"{rid}: {e.get('name')}: {f} {vals!r} not allowed")
        if not e.get("bases"):
            out.append(f"{rid}: {e.get('name')}: no base models listed")
    answered = [v["result"] for v in doc.get("values") or []]
    drawn = [c["result"] for c in cards]
    if sorted(answered) != sorted(drawn):
        out.append(f"{rid}: values answered {sorted(answered)} differ from the cards drawn {sorted(drawn)}")
    ids = {d["id"] for d in first.get("defences") or []}
    if len(match_line or []) != len(doc.get("defences") or []):
        out.append(f"{rid}: {len(match_line or [])} match lines for {len(doc.get('defences') or [])} defence entries")
    for ids_ in match_line or []:
        for i in ids_ or []:
            if i not in ids:
                out.append(f"{rid}: match {i!r} is not a first-pass defence")
    used = [i for ids_ in match_line or [] for i in ids_ or []]
    for i in {x for x in used if used.count(x) > 1}:
        out.append(f"{rid}: first-pass defence {i!r} matched twice")
    return out


# ---- agreement --------------------------------------------------------------------------------------------------

def cohen_kappa(pairs: list[tuple]) -> float | None:
    """Cohen's kappa over (first, second) pairs; missing values are a category. None when chance agreement is 1
    (every value in one category), where kappa is undefined and exact agreement is reported alone."""
    n = len(pairs)
    if not n:
        return None
    pairs = [(MISSING if a in (None, "") else a, MISSING if b in (None, "") else b) for a, b in pairs]
    po = sum(a == b for a, b in pairs) / n
    ca, cb = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    pe = sum(ca[c] * cb[c] for c in set(ca) | set(cb)) / (n * n)
    return None if pe >= 1 else (po - pe) / (1 - pe)


def jaccard(a, b) -> float:
    a, b = set(a or []), set(b or [])
    return 1.0 if not a and not b else len(a & b) / len(a | b)


def _number(s):
    t = str(s).strip().replace(",", "").replace("−", "-").replace("%", "").replace("+", "").strip()
    m = re.fullmatch(r"-?\d+(\.\d+)?", t)
    if not m:
        return None, None
    return float(t), len(m.group(1)) - 1 if m.group(1) else 0


def values_agree(first, second) -> bool:
    """Exact match within rounding: both printed numbers rounded to the coarser of their two precisions."""
    a, da = _number(first)
    b, db = _number(second)
    if a is None or b is None:
        return re.sub(r"\s+", "", str(first)) == re.sub(r"\s+", "", str(second))
    d = min(da, db)
    return round(a, d) == round(b, d)


def agreement(first: dict, second: dict, matches: dict) -> tuple[list[dict], list[dict], list[dict]]:
    """Agreement between the first coding (record files) and the second pass, before any disagreement is resolved.
    first: report id -> record; second: report id -> second-pass document; matches: report id -> list (one entry per
    second-pass defence, in order) of first-pass defence ids. Returns (field rows, per-report rows, disagreements)."""
    cat = defaultdict(list)          # field -> [(first, second)]
    multi = defaultdict(list)        # field -> [jaccard]
    multi_exact = defaultdict(list)
    vals, set_j, dis, per_report = [], [], [], []
    for rid in sorted(second):
        rec, sp, mt = first[rid], second[rid], matches[rid]
        cat["role"].append((rec["report"]["role"], sp["role"]))
        if rec["report"]["role"] != sp["role"]:
            dis.append({"report": rid, "unit": "report", "field": "role", "first": rec["report"]["role"], "second": sp["role"]})
        for q in QUANTITIES:
            a, b = rec["coverage"].get(q), sp["coverage"].get(q)
            cat[f"coverage.{q}"].append((a, b))
            if a != b:
                dis.append({"report": rid, "unit": "coverage", "field": q, "first": a, "second": b})
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        if len(mt) != len(sp["defences"]):
            raise ValueError(f"{rid}: {len(mt)} match lines for {len(sp['defences'])} second-pass defences")
        matched = set()
        for entry, ids in zip(sp["defences"], mt):
            for did in ids or []:
                if did not in dfs:
                    raise ValueError(f"{rid}: match {did!r} is not a first-pass defence")
                matched.add(did)
                d = dfs[did]
                for f in DEF_CATEGORICAL:
                    cat[f"defence.{f}"].append((d.get(f), entry.get(f)))
                    if d.get(f) != entry.get(f):
                        dis.append({"report": rid, "unit": did, "field": f, "first": d.get(f), "second": entry.get(f)})
                for f in DEF_MULTI:
                    j = jaccard(d.get(f), entry.get(f))
                    multi[f"defence.{f}"].append(j)
                    multi_exact[f"defence.{f}"].append(set(d.get(f) or []) == set(entry.get(f) or []))
                    if j < 1:
                        dis.append({"report": rid, "unit": did, "field": f, "first": ";".join(d.get(f) or []),
                                    "second": ";".join(entry.get(f) or [])})
        # defence set at the method level: second-pass entries against first-pass groups; first-pass defences no
        # entry matched are grouped by name (with the base model removed) so one missed method counts once
        unmatched_first = [d for i, d in dfs.items() if i not in matched]
        groups = {re.sub(r"\s+on\s+.*$", "", d["name"]).strip().lower() for d in unmatched_first}
        n_entries = len(sp["defences"])
        n_matched_entries = sum(1 for ids in mt if ids)
        j = n_matched_entries / (n_entries + len(groups)) if (n_entries + len(groups)) else 1.0
        set_j.append(j)
        for d in unmatched_first:
            dis.append({"report": rid, "unit": d["id"], "field": "defence set", "first": d["name"], "second": "(not coded)"})
        for entry, ids in zip(sp["defences"], mt):
            if not ids:
                dis.append({"report": rid, "unit": "second pass", "field": "defence set", "first": "(not coded)", "second": entry["name"]})
        res = {r["id"]: r for r in rec.get("results") or []}
        n_ok = 0
        for v in sp.get("values") or []:
            r = res.get(v["result"])
            if r is None:
                raise ValueError(f"{rid}: value card {v['result']!r} is not a first-pass result")
            ok = values_agree(r["value"], v["value"])
            n_ok += ok
            vals.append(ok)
            if not ok:
                dis.append({"report": rid, "unit": v["result"], "field": "value", "first": r["value"], "second": v["value"]})
        per_report.append({"report": rid, "first_defences": len(dfs), "second_entries": n_entries,
                           "matched_entries": n_matched_entries, "unmatched_first_groups": len(groups),
                           "defence_set_jaccard": round(j, 3), "values": len(sp.get("values") or []), "values_agree": n_ok})
    rows = []

    def add(ftype, field, n, exact=None, kappa=None, jac=None):
        ok = True
        if exact is not None and exact < EXACT_MIN:
            ok = False
        if kappa is not None and kappa < KAPPA_MIN:
            ok = False
        if jac is not None and jac < JACCARD_MIN:
            ok = False
        rows.append({"field_type": ftype, "field": field, "pairs": n,
                     "exact_agreement": "" if exact is None else round(exact, 3),
                     "kappa": "" if kappa is None else round(kappa, 3),
                     "mean_jaccard": "" if jac is None else round(jac, 3),
                     "meets_threshold": "yes" if ok else "no", "action": "none" if ok else "re-code this field in full"})
    for f, pairs in cat.items():
        add("categorical", f, len(pairs), sum(a == b for a, b in pairs) / len(pairs), cohen_kappa(pairs))
    allc = [p for ps in cat.values() for p in ps]
    add("categorical (pooled)", "all categorical fields", len(allc), sum(a == b for a, b in allc) / len(allc), cohen_kappa(allc))
    rows[-1]["action"] = "summary row (the rule applies field by field)"
    for f, js in multi.items():
        add("multi-label", f, len(js), sum(multi_exact[f]) / len(js), None, sum(js) / len(js))
    add("multi-label", "defence set (method level, per report)", len(set_j), None, None, sum(set_j) / len(set_j))
    add("numeric", "printed values re-read from their addresses", len(vals), sum(vals) / len(vals) if vals else 0)
    return rows, per_report, dis

In [ ]:
%%writefile tests/test_extract.py
"""Tests for src/extract.py: the seeded sample, value cards, tables, cross-record checks and agreement statistics."""
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
extract = importlib.import_module("extract")
schema_mod = importlib.import_module("schema")


def _rec(rid="r_a", slug="s_a"):
    return {"report": {"id": rid, "slug": slug, "role": "proposes", "read": "F"},
            "coverage": {q: "not_reported" for q in extract.QUANTITIES} | {"R1": "direct"},
            "defences": [{"id": "d1", "name": "Def", "origin": "this_report", "base_checkpoint": "M", "axis1": ["chat_roles"],
                          "axis2": ["sft"], "isolation": "no_ablation", "signal_dependence": "not_reported",
                          "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2}],
            "evaluations": [{"id": "e1", "defence_id": "d1", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3},
                            {"id": "e0", "defence_id": "none:M", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3}],
            "results": [{"id": f"x{i}", "evaluation_id": "e1" if i % 2 else "e0", "quantity": "R1", "coverage": "direct",
                         "metric_as_printed": "ASR", "value": f"{i}.5", "unit": "percent", "direction": "lower_better",
                         "qualifier": "exact", "value_source": "table", "page": 3} for i in range(9)],
            "comparisons": [{"id": "c1", "left": "e1", "right": "e0", "purpose": "W", "varies": "defence",
                             **{c: "matched" for c in schema_mod.CONDITIONS}, "page": 3}],
            "claims": [{"id": "k1", "text": "t", "location": "abstract", "anchor": "a", "jobs": ["R1"], "supported_by": ["c1"], "page": 1}]}


def test_sample_is_stable_and_twenty_percent():
    slugs = [f"s{i:03d}" for i in range(124)]
    s = extract.second_pass_sample(slugs)
    assert len(s) == 25 and len(set(s)) == 25
    dropped = [x for x in slugs if x != slugs[-1] and x not in s[:1]]
    s2 = extract.second_pass_sample(dropped)
    assert set(s[1:]) <= set(s2)                       # removing one study does not reorder the others
    assert extract.second_pass_sample(slugs) == s      # deterministic


def test_cards_hide_values_and_are_capped():
    cards = extract.value_cards(_rec())
    assert len(cards) == extract.CARDS_PER_REPORT
    assert all("value" not in c for c in cards)
    assert {c["arm"].startswith(("Def on", "no defence")) for c in cards} == {True}


def test_tables_have_global_ids_and_grades():
    sch = schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml")
    t = extract.build_tables({"r_a": _rec()}, sch, schema_mod.grade)
    assert t["defences"][0][0]["defence_uid"] == "r_a:d1"
    ev = {r["evaluation_uid"]: r for r in t["evaluations"][0]}
    assert ev["r_a:e1"]["measurement"] == "original" and ev["r_a:e0"]["measurement"] == "undefended"
    assert t["comparisons"][0][0]["grade"] == "A"
    assert t["claims"][0][0]["supported_by"] == "r_a:c1"


def test_cross_record_checks():
    recs = {"r_a": _rec()}
    assert extract.cross_record_problems(recs, ["s_a"]) == []
    probs = extract.cross_record_problems(recs, ["s_a", "s_b"], {"r_a": 2})
    assert any("s_b has no record" in p for p in probs)
    assert any("beyond the PDF's 2 pages" in p for p in probs)


def test_kappa_jaccard_and_rounding():
    assert extract.cohen_kappa([("a", "a"), ("b", "b")]) == 1.0
    assert extract.cohen_kappa([("a", "a"), ("a", "a")]) is None
    assert abs(extract.cohen_kappa([("a", "a"), ("a", "b"), ("b", "b"), ("b", "a")])) < 1e-9
    assert extract.jaccard(["x", "y"], ["x"]) == 0.5
    assert extract.values_agree("45.20%", "45.2") and extract.values_agree("+0.028", "0.028")
    assert not extract.values_agree("45.2", "45.3") and extract.values_agree("12", "12.0")


def test_agreement_counts_disagreements():
    rec = _rec()
    sp = {"report": "r_a", "role": "proposes", "coverage": dict(rec["coverage"]),
          "defences": [{"name": "Def", "bases": ["M"], "origin": "this_report", "axis1": ["chat_roles", "position"], "axis2": ["sft"],
                        "isolation": "ablation_present", "signal_dependence": "not_reported", "trains_instruction_pair": "no",
                        "trains_carry_through": "no"}],
          "values": [{"result": "x1", "value": "1.5"}, {"result": "x2", "value": "9"}]}
    rows, per, dis = extract.agreement({"r_a": rec}, {"r_a": sp}, {"r_a": [["d1"]]})
    by = {r["field"]: r for r in rows}
    assert by["defence.isolation"]["exact_agreement"] == 0.0
    assert by["defence.axis1"]["mean_jaccard"] == 0.5
    assert by["printed values re-read from their addresses"]["exact_agreement"] == 0.5
    assert {d["field"] for d in dis} == {"isolation", "axis1", "value"}
    assert extract.validate_second_pass(sp, schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml"),
                                        [{"result": "x1"}, {"result": "x2"}], rec, [["d1"]]) == []

In [ ]:
import schema, search, paths, extract
for m in (schema, search, paths, extract):
    importlib.reload(m)
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_extract.py"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout.strip() else t.stderr.strip()[-400:])

## Stage 2. The sample and the value cards

In [ ]:
import csv, datetime as dt
EXPECTED = ['ppa_arxiv2025', 'cacheprune_arxiv2025', 'actconsistency_arxiv2026', 'civ_arxiv2025', 'explosive_arxiv2026', 'loginject_arxiv2026', 'securecai_arxiv2026', 'intentguard_arxiv2025', 'tab_arxiv2026', 'agentsafeevo_arxiv2026', 'raised_arxiv2026', 'pictionary_arxiv2026', 'ipidefrepl_computers2026', 'rvr_arxiv2025', 'copa_arxiv2026', 'codt_arxiv2025', 'municipal_sauc2026', 'jia_critical_arxiv2025', 'tracer_emnlp2026', 'pismith_colm2026', 'advsim2real_arxiv2026', 'spotlighting_arxiv2024', 'autodojo_arxiv2026', 'ltbd_arxiv2026', 'advershield_jtsi2026']                      # the 25 reports the second coding was done on
man = search.read_csv(paths.REPORTS / "defences.csv")
included = sorted(m["slug"] for m in man if m.get("decision") == "include")
records = {}
for f in sorted(paths.RECORDS.glob("*.yaml")):
    r = schema.load_record(f)
    records[r["report"]["id"]] = r
slug2rid = {r["report"]["slug"]: rid for rid, r in records.items()}
sample = extract.second_pass_sample(included)
sample_ids = [slug2rid.get(s, "") for s in sample]
print(f"included studies: {len(included)}; sample: {len(sample)} (seed {extract.SEED}, share {extract.SHARE})")
missing = [s for s, r in zip(sample, sample_ids) if not r]
if missing or sorted(sample_ids) != sorted(EXPECTED):
    raise SystemExit(f"STOP: the drawn sample differs from the one coded (missing records: {missing}; "
                     f"drawn {sorted(sample_ids)})")
(paths.REPORTS / "second_pass_ids.txt").write_text("\n".join(f"{s}\t{r}" for s, r in zip(sample, sample_ids)) + "\n")
cards = {rid: extract.value_cards(records[rid]) for rid in sample_ids}
rows = [c for rid in sample_ids for c in cards[rid]]
search.write_csv(paths.REPORTS / "second_pass_cards.csv", rows, list(rows[0].keys()))
print(f"value cards: {len(rows)} (no values in the card file); sample matches the coded set")

## Stage 3. The second coding, as frozen before matching, and the matches

In [ ]:
Path("reports/second_pass").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

In [ ]:
%%writefile reports/second_pass/ppa_arxiv2025.yaml
report: ppa_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: not_reported, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: PPA (Polymorphic Prompt Assembling)
  origin: this_report
  bases: [GPT-3.5-Turbo, GPT-4-Turbo, Llama-3.3-70B, DeepSeek-V3]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: PPA system-prompt styles (PRE, ESD, EIBD, RIZD, WBR)
  origin: this_report
  bases: [GPT-3.5-Turbo]
  variants: [PRE, ESD, EIBD, RIZD, WBR]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_g4_escapechar, value: "1.40%"}
- {result: r_l3_adversaria, value: "0.00%"}
- {result: r_ds_naïveattac, value: "1.60%"}
- {result: r_l3_obfuscatio, value: "0.60%"}
- {result: r_g4_fakecomple, value: "5.80%"}
- {result: r_g35_adversaria, value: "0.20%"}

In [ ]:
%%writefile reports/second_pass/cacheprune_arxiv2025.yaml
report: cacheprune_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: proxy, R3safe: not_reported}
defences:
- name: CachePrune
  origin: this_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  axis1: [internal_intervention]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: CachePrune ablation variants (mask transfer, k, attribution loss, alpha, without phi, N)
  origin: this_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  variants: [mask transfer code/text, k=2, k=4, full attribution loss, alpha values, without phi, N=4, N=12]
  axis1: [internal_intervention]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Delimiting
  origin: other_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Datamarking
  origin: other_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Sandwich
  origin: other_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Encode_Base64
  origin: other_report
  bases: [LLama3-8B, Mistral-7B-Instruct-v0.3, Phi-3.5-mini-instruct]
  axis1: [encodings]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_m7_cp_squad_asr, value: "0.68"}
- {result: r_l3_cp_hotpot_f1, value: "10.97"}
- {result: r_m7_cp_squad_f1, value: "23.10"}
- {result: r_m7_cp_hotpot_f1, value: "13.32"}
- {result: r_m7_none_hotpot_f1c, value: "14.10"}
- {result: r_phi_cp_wild_asr, value: "1.89"}

In [ ]:
%%writefile reports/second_pass/actconsistency_arxiv2026.yaml
report: actconsistency_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: ACT (Activation Consistency Training)
  origin: this_report
  bases: [Qwen3-1.7B, Qwen3-8B, Phi-4-reasoning, GPT-OSS-20B, Gemma-4-E4B-it]
  axis1: [chat_roles]
  axis2: [distillation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: BCT (Bias-Augmented Consistency Training)
  origin: this_report
  bases: [Qwen3-1.7B, Qwen3-8B, Phi-4-reasoning, GPT-OSS-20B, Gemma-4-E4B-it]
  axis1: [chat_roles]
  axis2: [sft, distillation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecAlign (DPO) applied to a reasoning model
  origin: other_report
  bases: [Qwen3-8B]
  axis1: [chat_roles]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_t4_secalign_opiclean, value: "79.3"}
- {result: r_s_oss_none_sep, value: "73.2"}
- {result: r_s_q8_none_opiclean, value: "81.2"}
- {result: r_s_oss_none_mmlu, value: "83.1"}
- {result: r_s_q8_act_mmlu, value: "76.8"}
- {result: r_s_phi_act_sep, value: "31.7"}

In [ ]:
%%writefile reports/second_pass/civ_arxiv2025.yaml
report: civ_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: proxy, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: CIV (Contextual Integrity Verification), hard mask + gate
  origin: this_report
  bases: [Llama-3-8B-Instruct]
  axis1: [internal_intervention]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: CIV ablation variants
  origin: this_report
  bases: [Llama-3-8B-Instruct]
  variants: [hard mask only, no trust propagation, unsigned labels, KV trust disabled, soft mask]
  axis1: [internal_intervention]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_civ_elite, value: "0%"}
- {result: r_abl_full, value: "0%"}
- {result: r_abl_unsigned, value: "7.1%"}
- {result: r_none_sim, value: "100%"}
- {result: r_none_elite, value: "54%"}
- {result: r_civ_sim, value: "93.1%"}

In [ ]:
%%writefile reports/second_pass/explosive_arxiv2026.yaml
report: explosive_arxiv2026
role: mixed
coverage: {R1: direct, R2clean: not_reported, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Meta-SecAlign-70B (released)
  origin: other_report
  bases: [Llama-3.3-70B-Instruct]
  axis1: [chat_roles]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Meta-SecAlign-70B + EP-DPO continuation
  origin: this_report
  bases: [Llama-3.3-70B-Instruct]
  variants: [step-0050, step-0100, step-0150]
  axis1: [chat_roles]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_none_ep_turn2, value: "17.1"}
- {result: r_msa70_ipi, value: "0.0"}
- {result: r_none_ep_any, value: "27.5"}
- {result: r_msa70_ep_any, value: "11.8"}
- {result: r_msa70_ep_turn1, value: "0.0"}
- {result: r_t25_dpo, value: "9.59"}

In [ ]:
%%writefile reports/second_pass/loginject_arxiv2026.yaml
report: loginject_arxiv2026
role: benchmark
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Spotlighting (prompt hardening with XML delimiters and trust instruction)
  origin: other_report
  bases: [GPT-4o]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_exfil_none, value: "82.1"}
- {result: r_fab_spot, value: "52.1"}
- {result: r_exfil_spot, value: "45.3"}
- {result: r_all_none, value: "87.3"}
- {result: r_inst_spot, value: "49.8"}
- {result: r_ben_none, value: "94.2%"}

In [ ]:
%%writefile reports/second_pass/securecai_arxiv2026.yaml
report: securecai_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: SecureCAI (full)
  origin: this_report
  bases: [unnamed pre-trained 7B LLM]
  axis1: [no_signal]
  axis2: [sft, preference_optimisation]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecureCAI ablation variants
  origin: this_report
  bases: [unnamed pre-trained 7B LLM]
  variants: [w/o adaptive evolution, w/o DPO training, w/o unlearning, w/o security principles, w/o input sanitisation, constitutional SL only]
  axis1: [no_signal]
  axis2: [sft, preference_optimisation]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Standard CAI
  origin: other_report
  bases: [unnamed pre-trained 7B LLM]
  axis1: [no_signal]
  axis2: [sft, preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Instruction Hierarchy (priority-based prompt processing)
  origin: other_report
  bases: [unnamed pre-trained 7B LLM]
  axis1: [chat_roles]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_base_log, value: "87.3"}
- {result: r_cai_log, value: "42.7"}
- {result: r_base_mal, value: "79.4"}
- {result: r_ih_email, value: "35.2"}
- {result: r_base_clean, value: "91.5"}
- {result: r_ih_clean, value: "86.7"}

In [ ]:
%%writefile reports/second_pass/intentguard_arxiv2025.yaml
report: intentguard_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: IntentGuard (recovery mode)
  origin: this_report
  bases: [Qwen3-32B, gpt-oss-20B]
  axis1: [chat_roles]
  axis2: [prompting, fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: IntentGuard ablation variants
  origin: this_report
  bases: [Qwen3-32B, gpt-oss-20B]
  variants: [system prompt only, start-of-thinking prefilling only, end-of-thinking refinement only, format specification demonstration, conflict reasoning demonstration, dense embedding tracing]
  axis1: [chat_roles]
  axis2: [prompting, fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: SEP Defense (prompt engineering marking what to execute and what to process)
  origin: other_report
  bases: [Qwen3-32B, gpt-oss-20B]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_m2w_qwen_van_pair_asr, value: "1.000"}
- {result: r_adj_oss_van_pair_asr, value: "0.731"}
- {result: r_m2w_qwen_ig_gcg_util, value: "0.733"}
- {result: r_m2w_oss_ig_pair_util, value: "0.649"}
- {result: r_m2w_oss_sep_beam_util, value: "0.541"}
- {result: r_adj_oss_sep_ben, value: "0.649"}

In [ ]:
%%writefile reports/second_pass/tab_arxiv2026.yaml
report: tab_arxiv2026
role: benchmark
coverage: {R1: direct, R2clean: not_reported, R2attack: direct, R3ns: not_reported, R3cp: proxy, R3safe: not_reported}
defences:
- name: Spotlighting (sentinel markers around tool output)
  origin: other_report
  bases: [MiniMax-M2.7, Qwen-3.6-35B-A3B, GPT-5.4 mini]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: RUP (repeat user prompt / instruction reminder after tool output)
  origin: other_report
  bases: [MiniMax-M2.7, Qwen-3.6-35B-A3B, GPT-5.4 mini]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_mm_base_res, value: "36.0"}
- {result: r_qw_rup_dexec, value: "18.3"}
- {result: r_mm_rup_res, value: "31.1"}
- {result: r_qw_spot_res, value: "22.3"}
- {result: r_g54_spot_dexec, value: "28.4"}
- {result: r_g54_rup_dexec, value: "15.1"}

In [ ]:
%%writefile reports/second_pass/agentsafeevo_arxiv2026.yaml
report: agentsafeevo_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: not_reported, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: AgentSafeEvo (defender)
  origin: this_report
  bases: [Qwen3-8B]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: AgentSafeEvo ablation variants
  origin: this_report
  bases: [Qwen3-8B]
  variants: [w/o learnability weighting, w/o defender weighting, w/o attacker weighting]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Static-AT (fixed SFT attacker, defender-only RL)
  origin: this_report
  bases: [Qwen3-8B]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Spotlighting with Delimiting
  origin: other_report
  bases: [Qwen3-8B]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_spot_msb_nrp, value: "29.55"}
- {result: r_base_bfcl, value: "66.13"}
- {result: r_noaw_th, value: "40.63"}
- {result: r_static_ad_asr, value: "6.71"}
- {result: r_static_msb_asr, value: "38.64"}
- {result: r_ase_msb_nrp, value: "46.00"}

In [ ]:
%%writefile reports/second_pass/raised_arxiv2026.yaml
report: raised_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: RAISED (self-generation + self-distillation)
  origin: this_report
  bases: [Gemma-4-12B-it, Nemotron-3-Nano-30B-A3B, Qwen3.8-27B]
  axis1: [chat_roles]
  axis2: [distillation]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: RAISED ablation variants (data and objective)
  origin: this_report
  bases: [Gemma-4-12B-it]
  variants: [Glaive SFT, Glaive DPO, Glaive self-distillation, self-generated DPO]
  axis1: [chat_roles]
  axis2: [sft, preference_optimisation, distillation]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecAlign++
  origin: other_report
  bases: [Gemma-4-12B-it, Nemotron-3-Nano-30B-A3B, Qwen3.8-27B]
  variants: [main recipe, extended training (Nemotron)]
  axis1: [chat_roles]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Prompt Sandwiching (repeat user prompt)
  origin: other_report
  bases: [Gemma-4-12B-it, Nemotron-3-Nano-30B-A3B, Qwen3.8-27B]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Spotlighting
  origin: other_report
  bases: [Gemma-4-12B-it, Nemotron-3-Nano-30B-A3B, Qwen3.8-27B]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_nem_raised_dyu, value: "25.0"}
- {result: r_gem_sand_aduatk, value: "64.4"}
- {result: r_gem_sapp_adu, value: "77.3"}
- {result: r_gem_sand_dyasr, value: "56.1"}
- {result: r_gem_sapp_dyuatk, value: "46.2"}
- {result: r_gem_spot_adu, value: "80.4"}

In [ ]:
%%writefile reports/second_pass/pictionary_arxiv2026.yaml
report: pictionary_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: proxy, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Pictionary (untrusted content rendered as typographic images)
  origin: this_report
  bases: [Claude Opus 4.7, Claude Haiku 4.5, GPT-5.5, GPT-5.4 nano, GPT-5.4 mini, GPT-5.4, Qwen3.6-plus, Kimi-K2.6, Grok 4.3, Gemini 3.1 Pro, Gemini 3.1 Flash Lite]
  axis1: [encodings]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Pictionary variants (audio rendering, alternative visual styles, pagination)
  origin: this_report
  bases: [GPT-Audio Mini, Qwen3.5-Omni-Plus, several of the ten models]
  variants: [audio (text to speech), alternative visual styles, pagination]
  axis1: [encodings]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_art_kimi_img_u, value: "99"}
- {result: r_di_qwen_txt_asr, value: "96.4"}
- {result: r_audatt_qomni_aud_u, value: "93.4"}
- {result: r_di_glite_img_u, value: "100.0"}
- {result: r_di_nano_txt_asr, value: "14.3"}
- {result: r_di_kimi_txt_u, value: "76.0"}

In [ ]:
%%writefile reports/second_pass/ipidefrepl_computers2026.yaml
report: ipidefrepl_computers2026
role: re_measures
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Spotlighting (prompt structuring)
  origin: other_report
  bases: [GPT-5.4, GPT-5.4-mini, Claude Sonnet 4.6]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Repeat User Prompt (instruction reinforcement)
  origin: other_report
  bases: [GPT-5.4, GPT-5.4-mini, Claude Sonnet 4.6]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_g54m_none_asr, value: "3.8%"}
- {result: r_cs46_none_asr, value: "0.3%"}
- {result: r_g54m_rep_uua, value: "59.0%"}
- {result: r_g54m_rep_asr, value: "1.7%"}
- {result: r_g54_rep_uua, value: "55.2%"}
- {result: r_g54_none_bu, value: "56.2%"}

In [ ]:
%%writefile reports/second_pass/rvr_arxiv2025.yaml
report: rvr_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: RVR (Robustness via Referencing; tagged lines, referenced execution, tag filter)
  origin: this_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct, Llama3-70B-Instruct, Llama3.1-405B-Instruct, Qwen2-72B-Instruct, GPT-4o-mini, GPT-4o, GPT-4.1]
  axis1: [delimiters_markers]
  axis2: [prompting, fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: RVR variants (split only, dynamic tags, without in-context examples)
  origin: this_report
  bases: [GPT-4o-mini, GPT-4o, Llama3.1-8B-Instruct]
  variants: [split only, dynamic tags, without in-context examples]
  axis1: [delimiters_markers]
  axis2: [prompting, fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Sandwich
  origin: other_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct, Llama3-70B-Instruct, Llama3.1-405B-Instruct, Qwen2-72B-Instruct, GPT-4o-mini, GPT-4o, GPT-4.1]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Reminder
  origin: other_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct, Llama3-70B-Instruct, Llama3.1-405B-Instruct, Qwen2-72B-Instruct, GPT-4o-mini, GPT-4o, GPT-4.1]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Instructional
  origin: other_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct, Llama3-70B-Instruct, Llama3.1-405B-Instruct, Qwen2-72B-Instruct, GPT-4o-mini, GPT-4o, GPT-4.1]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Spotlight
  origin: other_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct, Llama3-70B-Instruct, Llama3.1-405B-Instruct, Qwen2-72B-Instruct, GPT-4o-mini, GPT-4o, GPT-4.1]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: StruQ
  origin: other_report
  bases: [Llama3-8B-Instruct, Qwen2-7B-Instruct, Llama3.1-8B-Instruct]
  axis1: [delimiters_markers]
  axis2: [sft]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_q2_none_util_sst2, value: "94.95"}
- {result: r_l31_struq_ignore, value: "1.11"}
- {result: r_l31_rvr_fakecom, value: "0.22"}
- {result: r_l31_rvr_util_sst2, value: "92.78"}
- {result: r_q2_rvr_util_sst2, value: "94.04"}
- {result: r_q2_none_util_triviaqa, value: "77.22"}

In [ ]:
%%writefile reports/second_pass/copa_arxiv2026.yaml
report: copa_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: not_reported, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: COPA (continual GRPO with margin-weighted replay)
  origin: this_report
  bases: [Meta-Llama-3.1-8B-Instruct, Mistral-7B-v0.3, Qwen2.5-7B]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: COPA ablation variants
  origin: this_report
  bases: [Meta-Llama-3.1-8B-Instruct]
  variants: [uniform replay, DPO objective]
  axis1: [no_signal]
  axis2: [adversarial_rl, preference_optimisation]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecAlign (embedded in the lifelong pipeline)
  origin: other_report
  bases: [Meta-Llama-3.1-8B-Instruct]
  axis1: [delimiters_markers]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_life_copa_ap, value: "0.850"}
- {result: r_mmlu_none, value: "0.332"}
- {result: r_life_copa_bwt, value: "+0.028"}
- {result: r_gpqa_none, value: "0.300"}
- {result: r_life_sa, value: "0.221"}
- {result: r_life_copa, value: "0.035"}

In [ ]:
%%writefile reports/second_pass/codt_arxiv2025.yaml
report: codt_arxiv2025
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Chain-of-Defensive-Thought (CoDT) prompting
  origin: this_report
  bases: [GPT-4o, GPT-3.5-turbo, Claude 3.5 Sonnet, Claude 3.5 Haiku, Claude 3 Sonnet, Claude 3 Haiku, Llama-3.1-8B-Instruct, Llama-3-8B-Instruct, Mistral-7B-Instruct-v0.3, Gemma-2-9b-it, Qwen2.5-7B-Instruct, deepseek-llm-7b-chat, OLMo-7B-0724-Instruct, Phi-3.5-mini-instruct, Yi-1.5-9B-Chat, Llama-2-7b-chat, Mistral-7B-Instruct-v0.2, Qwen2-7B-Instruct]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_nq_4_codt_asr, value: "28%"}
- {result: r_nq_14_codt_clean, value: "63%"}
- {result: r_nq_0_std_clean, value: "60%"}
- {result: r_rt_17_std_acc, value: "38%"}
- {result: r_rt_1_codt_clean, value: "63%"}
- {result: r_rt_17_codt_asr, value: "16%"}

In [ ]:
%%writefile reports/second_pass/municipal_sauc2026.yaml
report: municipal_sauc2026
role: re_measures
coverage: {R1: aggregate_only, R2clean: aggregate_only, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: D1 spotlighting via datamarking
  origin: other_report
  bases: [pooled over deepseek-v4-flash, qwen3.7-flash, gemini-3.5-flash-lite and mistral-small]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: D2 structured-query delimiting with an instruction hierarchy
  origin: other_report
  bases: [pooled over deepseek-v4-flash, qwen3.7-flash, gemini-3.5-flash-lite and mistral-small]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_o2_sq, value: "6"}
- {result: r_o5_none, value: "17"}
- {result: r_all_sq, value: "10.8"}
- {result: r_all_dm, value: "11.5"}
- {result: r_o2_none, value: "6"}
- {result: r_o5_sq, value: "15"}

In [ ]:
%%writefile reports/second_pass/jia_critical_arxiv2025.yaml
report: jia_critical_arxiv2025
role: re_measures
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: StruQ (released checkpoints)
  origin: other_report
  bases: [Llama-3-8B-Instruct, Llama-3-8B]
  axis1: [delimiters_markers]
  axis2: [sft]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecAlign (released checkpoints)
  origin: other_report
  bases: [Llama-3-8B-Instruct, Llama-3-8B]
  axis1: [delimiters_markers]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Instruction Hierarchy (as deployed in GPT-4o-mini)
  origin: other_report
  bases: [GPT-4o-mini]
  axis1: [chat_roles]
  axis2: [sft, preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r5, value: "0.46"}
- {result: r1, value: "0.52"}
- {result: r7, value: "36.42"}
- {result: r11, value: "0.34"}
- {result: r10, value: "0.45"}
- {result: r8, value: "0.65"}

In [ ]:
%%writefile reports/second_pass/tracer_emnlp2026.yaml
report: tracer_emnlp2026
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: proxy, R3safe: not_reported}
defences:
- name: TRACER (RAG-framed training, composite reward + ATRC)
  origin: this_report
  bases: [Qwen2.5-3B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [delimiters_markers]
  axis2: [adversarial_rl, distillation]
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: TRACER training-condition and objective variants
  origin: this_report
  bases: [Qwen2.5-3B-Instruct, Qwen2.5-7B-Instruct]
  variants: [follow-up-trained checkpoints, tool-trained, multi-framing RAG+tool, agent-trained, answer-only reward, composite reward without ATRC, 7B RAG epochs]
  axis1: [delimiters_markers]
  axis2: [adversarial_rl, distillation]
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_base7_tool, value: "36.6%"}
- {result: r_tool3_agent, value: "15.8%"}
- {result: r_multi3_tool, value: "21.7%"}
- {result: r_rag7e1_agent, value: "8.5%"}
- {result: r_abl_comp, value: "13.8"}
- {result: r_tool3_fu, value: "20.0%"}

In [ ]:
%%writefile reports/second_pass/pismith_colm2026.yaml
report: pismith_colm2026
role: re_measures
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Meta-SecAlign-8B
  origin: other_report
  bases: [Llama-3.1-8B-Instruct]
  axis1: [chat_roles]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Instruction Hierarchy (GPT-4o-mini)
  origin: other_report
  bases: [GPT-4o-mini]
  axis1: [chat_roles]
  axis2: [sft, preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: Sandwich
  origin: other_report
  bases: [Qwen3-4B-Instruct-2507]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Instructional
  origin: other_report
  bases: [Qwen3-4B-Instruct-2507]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_none_u, value: "0.74"}
- {result: r_sand_direct, value: "0.56"}
- {result: r_sand_pismith_at10, value: "0.99"}
- {result: r_sand_strat, value: "0.97"}
- {result: r_none_pair, value: "0.68"}
- {result: r_none_pismith, value: "0.92"}

In [ ]:
%%writefile reports/second_pass/advsim2real_arxiv2026.yaml
report: advsim2real_arxiv2026
role: proposes
coverage: {R1: proxy, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: AdvSim2Real executor (Stage 1 + 2, Robust iter 3)
  origin: this_report
  bases: [Qwen3.5-4B]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: AdvSim2Real checkpoints and ablation (Capability iter 1-3, Robust iter 1-2, Stage-2-only iter 1-3)
  origin: this_report
  bases: [Qwen3.5-4B]
  variants: [Capability iter 1, Capability iter 2, Capability iter 3, Robust iter 1, Robust iter 2, Stage-2-only iter 1, Stage-2-only iter 2, Stage-2-only iter 3]
  axis1: [no_signal]
  axis2: [adversarial_rl]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_base_v1, value: "51.11"}
- {result: r_robust3_clean, value: "81.33"}
- {result: r_cap3_v1, value: "57.11"}
- {result: r_robust3_v3, value: "54.67"}
- {result: r_base_v2, value: "48.00"}
- {result: r_robust3_v1, value: "62.89"}

In [ ]:
%%writefile reports/second_pass/spotlighting_arxiv2024.yaml
report: spotlighting_arxiv2024
role: proposes
coverage: {R1: direct, R2clean: direct, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Spotlighting via delimiting
  origin: this_report
  bases: [GPT-3.5-Turbo]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Spotlighting via datamarking
  origin: this_report
  bases: [GPT-3.5-Turbo, text-davinci-003, GPT-4]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Spotlighting via encoding (base64)
  origin: this_report
  bases: [GPT-3.5-Turbo, GPT-4]
  axis1: [encodings]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Defensive system instructions only
  origin: this_report
  bases: [GPT-3.5-Turbo]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r7, value: "0.0%"}
- {result: r1, value: "50%"}
- {result: r8, value: "1.8%"}
- {result: r3, value: "40%"}
- {result: r4, value: "0.00%"}
- {result: r5, value: "8.0%"}

In [ ]:
%%writefile reports/second_pass/autodojo_arxiv2026.yaml
report: autodojo_arxiv2026
role: benchmark
coverage: {R1: direct, R2clean: direct, R2attack: direct, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: Reminder (prompt-level)
  origin: other_report
  bases: [GPT-4o-mini, GPT-5.4-mini, Gemini-2.5-Flash, DeepSeek-v4-Flash, Claude-Haiku-4.5]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Spotlighting (prompt-level)
  origin: other_report
  bases: [GPT-4o-mini, GPT-5.4-mini, Gemini-2.5-Flash, DeepSeek-v4-Flash, Claude-Haiku-4.5]
  axis1: [delimiters_markers]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_ad_spot_hk_sa, value: "0.0"}
- {result: r_ad_none_g4m_aa, value: "52.4"}
- {result: r_ad_none_gem_cu, value: "63.2"}
- {result: r_ad_none_hk_cu, value: "70.2"}
- {result: r_ad_spot_g4m_sua, value: "43.4"}
- {result: r_ad_none_ds_sa, value: "22.6"}

In [ ]:
%%writefile reports/second_pass/ltbd_arxiv2026.yaml
report: ltbd_arxiv2026
role: proposes
coverage: {R1: direct, R2clean: proxy, R2attack: not_reported, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: LTBD (learnable trust-boundary delimiters, boundary + CE + preference)
  origin: this_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [delimiters_markers]
  axis2: [soft_tokens_frozen]
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: LTBD ablation variants (prefix + CE, boundary + CE)
  origin: this_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct]
  variants: [prefix + CE, boundary + CE]
  axis1: [delimiters_markers]
  axis2: [soft_tokens_frozen]
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: DefensiveToken
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [delimiters_markers]
  axis2: [soft_tokens_frozen]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: TextGrad (optimised defensive prompt)
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Reminder
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [no_signal]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: Sandwich
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [position]
  axis2: [prompting]
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: StruQ (LoRA and full fine-tuning)
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  variants: [StruQ-LoRA, StruQ-Full]
  axis1: [delimiters_markers]
  axis2: [sft]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
- name: SecAlign-LoRA
  origin: other_report
  bases: [Llama3-8B-Instruct, Llama3.1-8B-Instruct, Falcon3-7B-Instruct, Qwen2.5-7B-Instruct]
  axis1: [delimiters_markers]
  axis2: [preference_optimisation]
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
values:
- {result: r_l31_ltbd_alp_win, value: "30.98"}
- {result: r_f3_sec_tt_asr, value: "1.1"}
- {result: r_q25_dt_alp_asr, value: "0.96"}
- {result: r_l31_dt_tt_asr, value: "0.19"}
- {result: r_q25_struq_sep_asr, value: "2.0"}
- {result: r_f3_none_sep_asr, value: "78.49"}

In [ ]:
%%writefile reports/second_pass/advershield_jtsi2026.yaml
report: advershield_jtsi2026
role: proposes
coverage: {R1: proxy, R2clean: direct, R2attack: unclear, R3ns: not_reported, R3cp: not_reported, R3safe: not_reported}
defences:
- name: TAGD only (token-adaptive Gaussian smoothing at generation)
  origin: this_report
  bases: [Mistral-7B-Instruct-v0.3]
  axis1: [no_signal]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
- name: AdverShield-LLM without PLSA (TAGD + IACRS)
  origin: this_report
  bases: [Mistral-7B-Instruct-v0.3]
  axis1: [no_signal]
  axis2: [fixed_inference_rule]
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
values:
- {result: r_none, value: "74.2"}
- {result: r_tagd, value: "31.6"}

In [ ]:
%%writefile reports/second_pass_matches.yaml
# Second pass: which first-pass defence records each second-pass defence entry corresponds to, one line per entry
# in the order of reports/second_pass/<report_id>.yaml. Written after the second coding was finished and frozen,
# by comparing method names and base models only; it changes no coded value. [] = no first-pass record.
ppa_arxiv2025: [[ppa_g35, ppa_g4, ppa_l3, ppa_ds], []]
cacheprune_arxiv2025: [[cp_l3, cp_m7, cp_phi], [], [], [], [], []]
actconsistency_arxiv2026: [[act_q17, act_q8, act_phi, act_oss, act_gem], [bct_q17, bct_q8, bct_phi, bct_oss, bct_gem], [secalign_q8]]
civ_arxiv2025: [[civ_l3], [civ_unsigned_l3]]
explosive_arxiv2026: [[msa70], [epdpo]]
loginject_arxiv2026: [[spot]]
securecai_arxiv2026: [[], [secai_nosan], [cai], [ih]]
intentguard_arxiv2025: [[ig_qwen, ig_oss], [], [sep_qwen, sep_oss]]
tab_arxiv2026: [[spot_mm, spot_qw, spot_g54], [rup_mm, rup_qw, rup_g54]]
agentsafeevo_arxiv2026: [[ase], [nolw, nodw, noaw], [static], [spot]]
raised_arxiv2026: [[raised_gem, raised_nem], [], [sapp_gem, sapp_nem], [sand_gem, sand_nem], [spot_gem, spot_nem]]
pictionary_arxiv2026: [[pic_opus, pic_haiku, pic_g55, pic_nano, pic_mini, pic_qwen, pic_kimi, pic_grok, pic_gpro, pic_glite, pic_g54], [aud_gaud, aud_qomni]]
ipidefrepl_computers2026: [[spot_g54, spot_g54m, spot_cs46], [rep_g54, rep_g54m, rep_cs46]]
rvr_arxiv2025: [[rvr_l3, rvr_q2, rvr_l31], [], [], [], [], [], [struq_l3, struq_q2, struq_l31]]
copa_arxiv2026: [[copa_l31], [], [secalign_l31]]
codt_arxiv2025: [[codt_0, codt_1, codt_2, codt_3, codt_4, codt_5, codt_6, codt_7, codt_8, codt_9, codt_10, codt_11, codt_12, codt_13, codt_14, codt_15, codt_16, codt_17]]
municipal_sauc2026: [[dm], [sq]]
jia_critical_arxiv2025: [[struq_llama3_8bi_rel], [secalign_llama3_8bi_rel], []]
tracer_emnlp2026: [[rag3, rag7, t6, t11], [tool3, multi3, agent3, rag7e1, fu3, abl_ans, abl_comp]]
pismith_colm2026: [[msa8b], [], [sand_q], [instr_q]]
advsim2real_arxiv2026: [[robust3], [cap3]]
spotlighting_arxiv2024: [[spot_delimit_gpt35], [spot_datamark_gpt35, spot_datamark_text003, spot_datamark_gpt4], [spot_encode_gpt35], []]
autodojo_arxiv2026: [[rem_g4m, rem_g54m, rem_gem, rem_ds, rem_hk], [spot_g4m, spot_g54m, spot_gem, spot_ds, spot_hk]]
ltbd_arxiv2026: [[ltbd_l3, ltbd_l31, ltbd_f3, ltbd_q25], [pre_l3, bce_l3, pre_l31, bce_l31], [dt_l3, dt_l31, dt_f3, dt_q25], [], [], [], [struq_l3, struq_l31, struq_f3, struq_q25], [sec_l3, sec_l31, sec_f3, sec_q25]]
advershield_jtsi2026: [[tagd], []]

## Stage 4. Agreement per field type, before any disagreement is resolved

In [ ]:
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
second = {rid: schema.load_yaml(paths.REPORTS / "second_pass" / f"{rid}.yaml") for rid in sample_ids}
matches = schema.load_yaml(paths.REPORTS / "second_pass_matches.yaml")
problems = []
for rid in sample_ids:
    problems += extract.validate_second_pass(second[rid], S1, cards[rid], records[rid], matches.get(rid))
for p in problems:
    print("PROBLEM:", p)
if problems:
    raise SystemExit("STOP: second-pass files do not validate")
rows, per_report, dis = extract.agreement({rid: records[rid] for rid in sample_ids}, second, matches)
search.write_csv(paths.REPORTS / "agreement_04.csv", rows, list(rows[0].keys()))
search.write_csv(paths.REPORTS / "second_pass_reports_04.csv", per_report, list(per_report[0].keys()))
search.write_csv(paths.REPORTS / "second_pass_disagreements_04.csv", dis, ["report", "unit", "field", "first", "second"])
print(f"{'field':<46} {'type':<20} {'pairs':>5} {'exact':>6} {'kappa':>6} {'jacc':>6}  meets")
for r in rows:
    print(f"{r['field']:<46} {r['field_type']:<20} {r['pairs']:>5} {str(r['exact_agreement']):>6} {str(r['kappa']):>6} "
          f"{str(r['mean_jaccard']):>6}  {r['meets_threshold']}")
from collections import Counter
print("disagreements by field:", dict(Counter(d["field"] for d in dis)))

## Stage 5. Decision record and tests

In [ ]:
names = {"role": "role", "coverage.R1": "coverage R1", "coverage.R2clean": "coverage R2clean", "coverage.R2attack": "coverage R2attack",
         "coverage.R3ns": "coverage R3ns", "coverage.R3cp": "coverage R3cp", "coverage.R3safe": "coverage R3safe",
         "defence.origin": "defence origin", "defence.isolation": "defence isolation", "defence.signal_dependence": "signal dependence",
         "defence.trains_instruction_pair": "trains_instruction_pair", "defence.trains_carry_through": "trains_carry_through",
         "defence.axis1": "axis 1", "defence.axis2": "axis 2", "defence set (method level, per report)": "defence set",
         "printed values re-read from their addresses": "printed values"}
scored = [r for r in rows if not r["field_type"].endswith("(pooled)")]
below = [names[r["field"]] for r in scored if r["meets_threshold"] == "no"]
meets = [names[r["field"]] for r in scored if r["meets_threshold"] == "yes"]
vals = next(r for r in rows if r["field"].startswith("printed values"))
n_ok = sum(p["values_agree"] for p in per_report)
status = (f"AGREEMENT BELOW THRESHOLD ON {len(below)} FIELDS" if below else "AGREEMENT MEETS THRESHOLD ON EVERY FIELD") + " (consistency check, one coder)"
lines = [f"decision_04sp: {status}", f"this step: {dt.date.today().isoformat()}",
         f"sample: {len(sample_ids)} of {len(included)} included studies (seed {extract.SEED}, hash rank), listed in reports/second_pass_ids.txt",
         f"printed values re-read from their addresses: {n_ok} of {vals['pairs']} agree",
         "meets the threshold: " + ", ".join(meets),
         "below the threshold, to be re-coded in full: " + (", ".join(below) or "none"),
         "note: kappa is low where one code holds almost every pair (R3ns and R3safe agree in 24 of 25 reports); the rule is applied as written",
         "next: amend the coding rules where the disagreements show a gap, re-code these fields in every record (notebook 04r), "
         "code the last two studies (batch 04l), then notebook 04_full_extraction"]
(paths.REPORTS / "decision_04sp.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04 second pass: seeded 20 percent sample of 25 studies coded again from the PDFs, agreement per field type before resolution"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)